In [ ]:
import os
import commons as c

import pandas as pd
from scipy.stats import chi2_contingency, kruskal, friedmanchisquare, wilcoxon

# Statistical Analysis

In [ ]:
csv_path = 'results/dataframes/results_balanced.csv'
df_balanced = pd.read_csv(csv_path, dtype=c.type_dict)

csv_path = 'results/dataframes/results_equiv.csv'
df_equiv = pd.read_csv(csv_path, dtype=c.type_dict)

csv_path = 'results/dataframes/results_normal.csv'
df_normal = pd.read_csv(csv_path, dtype=c.type_dict)

df = pd.concat([df_equiv, df_normal], ignore_index=True)

In [ ]:
df['threshold_metric'] = list(zip(df['threshold'], df['metric']))

# Separate categorical and numerical columns
categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

for cat in ['threshold', 'metric', 'threshold_metric']:
    categorical_columns.remove(cat)  

print(categorical_columns)
print(numerical_columns)

In [ ]:
def ratio_correct_counts(df, col):
    if 'threshold_metric' not in df.columns or 'correctness' not in df.columns:
        raise ValueError("The DataFrame must contain 'threshold_metric' and 'correctness' columns.")
    
    # Step 1: Group by number of qubits and threshold metric, then count the correct values
    total_counts = (
        df.groupby([col, 'threshold_metric'])
        .size()
        .reset_index(name='total_count')  # Total count for each group
    )
    
    correct_counts = (
        df[df['correctness'] == True]  # Filter only correct rows
        .groupby([col, 'threshold_metric'])
        .size()
        .reset_index(name='correct_count')  # Reset index and name the count column
    )
    
    correct_counts = pd.merge(correct_counts, total_counts, on=[col, 'threshold_metric'], how='left')
    
    # Step 4: Calculate the ratio of correct_count to total_count
    correct_counts['correct_ratio'] = correct_counts['correct_count'] / correct_counts['total_count']
    
    return correct_counts

### Friedman Chi Square test

In [ ]:
all_cats = numerical_columns  + categorical_columns

#['Input', 'Input_type', 'Algorithm', 'Operator', 'Gate', 'Gate_type', 'Relative_position', 'Output_type', 'hardware']
#['gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Position', 'Qubits']

for cat in categorical_columns: #['Gate_type']: # ['hardware', 'Operator', 'Algorithm', 'Gate']:   
    print(f"For category {cat}:")

    # Ranked DataFrame from previous steps
    correct_counts = ratio_correct_counts(df, cat)
    grouped_ranks = correct_counts.groupby(cat)['correct_ratio'].apply(list)
    
    print(grouped_ranks)
    
    l = len(correct_counts.groupby(cat))
    if l > 2:
        stat, p_value = friedmanchisquare(*grouped_ranks)
        print(f"P-value from Friedman test: {p_value}")
        print(f"Stat from Friedman test: {stat}")
        
    else:
        stat, p_value = wilcoxon(*grouped_ranks)
        print(f"P-value from Wilcoxon test: {p_value}")
        print(f"Stat from Wilcoxon test: {stat}")
        
    if p_value < 0.05:
        print("Result: Significant association between 'cat' and 'threshold_metric'.")
    else:
        print("Result: No significant association between 'cat' and 'threshold_metric'.")
    print(f"================================================================")
    

In [ ]:
all_cats = numerical_columns  + categorical_columns

#['Input', 'Input_type', 'Algorithm', 'Operator', 'Gate', 'Gate_type', 'Relative_position', 'Output_type', 'hardware']
#['gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Position', 'Qubits']

for cat in numerical_columns:   
    print(f"For category {cat}:")

    # Ranked DataFrame from previous steps
    correct_counts = ratio_correct_counts(df, cat)
    grouped_ranks = correct_counts.groupby(cat)['correct_ratio'].apply(list)
    
    l = len(correct_counts.groupby(cat))
    if l > 2:
        
        try:
            stat, p_value = friedmanchisquare(*grouped_ranks)
            print(f"P-value from Friedman test: {p_value}")
            print(f"Stat from Friedman test: {stat}")
        except Exception as e:
            print(f'Error processing column {cat}: {str(e)}')
        
    else:
        stat, p_value = wilcoxon(*grouped_ranks)
        print(f"P-value from Wilcoxon test: {p_value}")
        print(f"Stat from Wilcoxon test: {stat}")
        
    if p_value < 0.05:
        print("Result: Significant association between 'cat' and 'threshold_metric'.")
    else:
        print("Result: No significant association between 'cat' and 'threshold_metric'.")
    print(f"================================================================")
    

In [ ]:
x = [72, 96, 88, 92, 74, 76, 82]
y = [120, 120, 132, 120, 101, 96, 112]
z = [76, 95, 104, 96, 84, 72, 76]
res = friedmanchisquare(x, y, z)
print("Friedman")
print(res.pvalue)
print(res.statistic)
res = kruskal(x, y, z)
print("Kruskal")
print(res.pvalue)
print(res.statistic)
print('')


x = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
y = [1, 20, 30, 40, 50, 60, 70, 80, 90, 100]
z = [5, 25, 35, 45, 55, 65, 75, 85, 95, 105]

array = [x, y, z]
transposed_array = [[row[i] for row in array] for i in range(len(x))]

res = friedmanchisquare(*array)
print("Friedman")
print(res.pvalue)
print(res.statistic)
res = kruskal(*transposed_array)
print("Kruskal")
print(res.pvalue)
print(res.statistic)

print('')
res = friedmanchisquare(*transposed_array)
print("Friedman")
print(res.pvalue)
print(res.statistic)
res = kruskal(*array)
print("Kruskal")
print(res.pvalue)
print(res.statistic)

### Kruskal-Wallis test

In [ ]:
all_cats = numerical_columns  + categorical_columns

#['Input', 'Input_type', 'Algorithm', 'Operator', 'Gate', 'Gate_type', 'Relative_position', 'Output_type', 'hardware']
#['gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Position', 'Qubits']
significant = []
non_significant = []

for cat in all_cats: #['Gate_type']: # ['hardware', 'Operator', 'Algorithm', 'Gate']:   
    print(f"For category {cat}:")

    # Ranked DataFrame from previous steps
    correct_counts = ratio_correct_counts(df, cat)
    grouped_ranks = correct_counts.groupby(cat)['correct_ratio'].apply(list)
    
    # Perform Kruskal-Wallis test
    stat, p_value = kruskal(*grouped_ranks)
    
    print(f"P-value from Kruskal-Wallis test: {p_value}")
    print(f"Stat from Kruskal-Wallis test: {stat}")
        
    if p_value < 0.05:
        print(f"Result: Significant association between {cat} and 'threshold_metric'.")
        significant.append(cat)
    else:
        print(f"Result: No significant association between {cat} and 'threshold_metric'.")
        non_significant.append(cat)
    print(f"================================================================")
    
    
print(f"Significant association between the threshold_metric selection and {significant}.")
print(f"Non-Significant association between the threshold_metric and {non_significant}.")
    